## `Search Engine -> DuckDuckGo`

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()

search.invoke("Obama")

### `Search Engine -> DuckDuckGo custom`

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(region="xa-ar", max_results=2)

search = DuckDuckGoSearchResults(api_wrapper=wrapper, output_format="list")

outupt = search.invoke("Obama")

In [ ]:
outupt[0]

### `Search Engine -> DuckDuckGo Agent`

In [6]:
from dotenv import load_dotenv
from langchain_classic.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_classic.agents import initialize_agent, AgentType, load_tools

_ = load_dotenv(override=True)

def DuckDuckGoSearchTool() -> Tool:
    search = DuckDuckGoSearchResults(output_format="list")
    tool = Tool(
        name="Google Search Snippets",
        description="Search Google for recent results.",
        func=search.invoke,
    )
    return tool


llm = ChatOpenAI(name="gpt-5-nano-2025-08-07", temperature=0.5)

# Initialize an agent
agent = initialize_agent(
    tools=[DuckDuckGoSearchTool()],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

response = agent.invoke("What is the capital of Canada? Provide recent information.")
response



> Entering new AgentExecutor chain...
I should use the Google Search Snippets tool to find the most recent information on the capital of Canada.
Action: Google Search Snippets
Action Input: 'capital of Canada'
Observation: [{'snippet': "Canada 's capital is Ottawa and its three largest metropolitan areas are Toronto, Montreal, and Vancouver.", 'title': 'Canada - Wikipedia', 'link': 'https://en.wikipedia.org/wiki/Canada'}, {'snippet': 'As Canada ’s largest east coast port, deep-water and ice-free, the capital , Halifax, has played an important role in Atlantic trade and defence and is home to Canada ’s largest naval base.', 'title': 'Discover Canada - Canada ’s Regions - Canada .ca', 'link': 'https://www.canada.ca/en/immigration-refugees-citizenship/corporate/publications-manuals/discover-canada/read-online/canadas-regions.html'}, {'snippet': 'Moreover, like every decent capital city, Ottawa is a city to learn and explore the history of Canada , with a room for some well-known urban a

{'input': 'What is the capital of Canada? Provide recent information.',
 'output': 'Ottawa'}

---

## `Gmail Access`

In [ ]:
from dotenv import load_dotenv
from langchain_classic.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)
from langchain_google_community import GmailToolkit
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.agents import create_openai_functions_agent, AgentExecutor

load_dotenv()

credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://mail.google.com/"],
)

api_resource = build_resource_service(credentials=credentials)
toolkit = GmailToolkit(api_resource=api_resource)
tools = toolkit.get_tools()

## Initialize LLM & Agent
llm = ChatOpenAI(name="gpt-5-nano-2025-08-07", temperature=0.5)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an assistant that helps manage Gmail. "
            "Only use Gmail tools when the user asks about emails, inbox, or sending messages. "
            "Otherwise, answer normally."
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

response = agent_executor.invoke(
    {"input": "Give me summary for the last 1 emails."}
)

print(response)

----

## `Calendar Agent`

In [ ]:
from dotenv import load_dotenv
from langchain_classic.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_google_community import CalendarToolkit
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.agents import create_openai_functions_agent, AgentExecutor

load_dotenv()

toolkit = CalendarToolkit()
tools = toolkit.get_tools()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=661211293190-fhes3mnvpfe5lo1109d1akgp5gqrodja.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A62413%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=N8l4tMC3ALGD4G6Tg29268JFMHHOf1&access_type=offline


In [ ]:
# Use patched_tools
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

response = agent_executor.invoke(
    {"input": "Create a meeting with Osama tomorrow at 3 PM called 'Project Sync'"}
)

print(response)

----